# Model Size Testing (Local Hugging Face Models)

For benchmarking different model sizes, from 500M to 10B parameter size.

## Imports and Environment Checks

In [ ]:
import time
import torch
from transformers import pipeline

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

## Shared Generation Function

In [ ]:
def generate_text(pipe, prompt, max_length=120):
    result = pipe(
        prompt,
        max_length=max_length,
        do_sample=True,
        top_p=0.95
    )
    return result[0]['generated_text']

## 
Model Size Configuration (500M to 10B+)

In [ ]:
MODEL_TIERS = [
    {"label": "~0.5B", "params_b": 0.5, "model_id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"label": "~1.1B", "params_b": 1.1, "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"},
    {"label": "~3B", "params_b": 3.0, "model_id": "Qwen/Qwen2.5-3B-Instruct"},
    {"label": "~7B", "params_b": 7.0, "model_id": "Qwen/Qwen2.5-7B-Instruct"},
    {"label": "~10B+", "params_b": 14.0, "model_id": "Qwen/Qwen2.5-14B-Instruct"},
]

for m in MODEL_TIERS:
    print(f"{m['label']:>7} | {m['params_b']:>4}B params | {m['model_id']}")

## Prompt Setup

In [ ]:
base_prompt = "What is the nature of grief and revenge?"
persona_suffix = " Answer as if you are Hamlet from Shakespeare's play."
prompt = base_prompt + persona_suffix

print(prompt)

## Single Model Test

Set `MODEL_INDEX` to pick one size tier first.

In [ ]:
MODEL_INDEX = 0  # 0..4
selected = MODEL_TIERS[MODEL_INDEX]

print(f"Loading: {selected['model_id']} ({selected['label']})")
pipe = pipeline(
    "text-generation",
    model=selected['model_id'],
    device_map="auto",
    local_files_only=True
)

start = time.perf_counter()
generated_text = generate_text(pipe, prompt, max_length=120)
elapsed = time.perf_counter() - start

print(f"Inference time: {elapsed:.2f}s")
print('Generated Text:\n')
print(generated_text)

## Batch Benchmark Across Sizes

Runs every configured model tier and records timing + sample output.

In [ ]:
results = []

for model_cfg in MODEL_TIERS:
    model_id = model_cfg['model_id']
    label = model_cfg['label']
    
    print(f"\n=== Testing {label}: {model_id} ===")
    
    try:
        load_start = time.perf_counter()
        pipe = pipeline(
            "text-generation",
            model=model_id,
            device_map="auto",
            local_files_only=True
        )
        load_time = time.perf_counter() - load_start

        gen_start = time.perf_counter()
        output = generate_text(pipe, prompt, max_length=120)
        gen_time = time.perf_counter() - gen_start

        record = {
            'label': label,
            'params_b': model_cfg['params_b'],
            'model_id': model_id,
            'status': 'ok',
            'load_time_s': round(load_time, 2),
            'gen_time_s': round(gen_time, 2),
            'sample_output': output[:500]
        }

        print(f"Load: {record['load_time_s']}s | Generate: {record['gen_time_s']}s")

    except Exception as exc:
        record = {
            'label': label,
            'params_b': model_cfg['params_b'],
            'model_id': model_id,
            'status': 'failed',
            'error': str(exc)
        }

        print(f"FAILED: {exc}")

    results.append(record)

## Results Summary

In [ ]:
from pprint import pprint

pprint(results)

## Saving Results to a JSON

In [ ]:
import json

output_path = 'model_size_benchmark_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved results to: {output_path}")